# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssof-Essam/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract (5 plain-words answers):**

1. **One row = one content item (page)**, aggregated over a prior 90-day feature window ending at the decision date.

2. **Tables**: `dim_content` (metadata) + `fact_content_daily_performance` (daily metrics, partitioned by `month=YYYY-MM`).

3. **Time window**: Iterate on mid-panel month **`month=2026-03`** as the outcome month. Feature window = 90 days before March 1 (Dec 2025–Feb 2026). Label = `trend_direction == "down"` in March 2026 (current-window proxy, consistent with w01/w02).

4. **Label/proxy**: `is_declining_label = (trend_direction == "down")` from the March 2026 slice.

5. **Excluded**: `trend_direction`, `trend_pct`, any target-window metrics (`*_last30`, `*_last7`) — label-derived, never features.

In [ ]:
# --- Token loader (Colab Secrets or .env) ---
import os
import sys

def get_hf_token():
    # 1. Try Colab secrets first
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    
    # 2. Fall back to .env file (local dev)
    from pathlib import Path
    env_path = Path(".env")
    if env_path.exists():
        from dotenv import load_dotenv
        load_dotenv()
        token = os.getenv("HF_TOKEN")
        if token:
            return token
    
    # 3. Fall back to env var (already set in shell)
    token = os.getenv("HF_TOKEN")
    if token:
        return token
    
    raise RuntimeError("HF_TOKEN not found. Set in Colab Secrets or .env file.")

HF_TOKEN = get_hf_token()
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN loaded")

# --- Use datasets library (handles revision + token correctly) ---
from datasets import load_dataset
import pandas as pd

# Load March 2026 fact table (streaming) and dim_content
fact_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)
dim_content_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

# Filter to month=2026-03 and convert to pandas for small sample
march_items = []
for row in fact_ds:
    if row.get('month') == '2026-03':
        march_items.append(row)
    if len(march_items) >= 5:
        break

fact_df = pd.DataFrame(march_items)

# Get dim_content as dict for join
dim_content_map = {}
for row in dim_content_ds:
    dim_content_map[row['content_hash_id']] = row
    if len(dim_content_map) > 10000:  # limit memory
        break

# Merge sample
def enrich(row):
    d = dim_content_map.get(row['content_hash_id'], {})
    return pd.Series({
        'content_created_at': d.get('content_created_at'),
        'days_since_last_update': d.get('days_since_last_update'),
        'word_count': d.get('word_count'),
        'content_type': d.get('content_type'),
        'main_intent': d.get('main_intent'),
    })

enriched = fact_df.apply(enrich, axis=1)
df_sample = pd.concat([fact_df, enriched], axis=1)

show_cols = ['report_date', 'client_hash_id', 'content_hash_id',
             'gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
             'ga4_sessions', 'ga4_engagement_rate',
             'ga4_data_available', 'gsc_data_available',
             'content_created_at', 'days_since_last_update',
             'word_count', 'content_type', 'main_intent']
print("Sample rows (month=2026-03, joined with dim_content):")
print(df_sample[show_cols].head(5).to_string())


HF_TOKEN loaded


/home/youssof/Flyrank-Internship/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Fields: feature / label / context / excluded

**Field classification for our Lane 2 slice:**

| Bucket | Columns | Why |
|--------|---------|-----|
| **Feature** | `log_impressions_90d`, `avg_position_90d` (excl 0), `ctr_90d`, `days_since_last_update`, `content_age_days`, `word_count` + `has_word_count`, `engagement_rate_90d`, `sessions_90d` | Aggregated from feature window (90 days before decision); knowable before we act |
| **Label/Proxy** | `is_declining_label` (`trend_direction == "down"` in March 2026) | Current-window proxy; what we predict |
| **Context** | `content_hash_id`, `client_hash_id`, `report_date` | Grouping, joining, splitting — never model features |
| **Excluded** | `trend_direction`, `trend_pct`, `gsc_impressions_last30`, `gsc_clicks_last30`, `ga4_sessions_last30`, any target-window metric | Label-derived / future information — leakage risk |

In [ ]:
# Show column -> bucket map for the joined frame
import pandas as pd

field_map = {
    # Features
    "log_impressions_90d": "feature",
    "avg_position_90d": "feature",
    "ctr_90d": "feature",
    "days_since_last_update": "feature",
    "content_age_days": "feature",
    "word_count": "feature",
    "has_word_count": "feature",
    "engagement_rate_90d": "feature",
    "sessions_90d": "feature",
    # Label / Proxy
    "is_declining_label": "label_proxy",
    # Context
    "content_hash_id": "context",
    "client_hash_id": "context",
    "report_date": "context",
    # Excluded (leakage)
    "trend_direction": "excluded",
    "trend_pct": "excluded",
    "gsc_impressions_last30": "excluded",
    "gsc_clicks_last30": "excluded",
    "ga4_sessions_last30": "excluded",
}

df_map = pd.DataFrame([{"column": k, "bucket": v} for k, v in field_map.items()])
print(df_map.to_string(index=False))


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries on `month=2026-03`, each with output visible.

In [ ]:
# Query 1: Grain check — one row = one (report_date, client_hash_id, content_hash_id)
from datasets import load_dataset
import pandas as pd

fact_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

duplicates = {}
count = 0
for row in fact_ds:
    if row.get('month') != '2026-03':
        continue
    key = (row['report_date'], row['client_hash_id'], row['content_hash_id'])
    duplicates[key] = duplicates.get(key, 0) + 1
    count += 1
    if count % 100000 == 0:
        print(f"  Processed {count} rows...")

dup_keys = [k for k, v in duplicates.items() if v > 1]
print(f"Grain check — duplicate (date, client, content) rows: {len(dup_keys)}")
if len(dup_keys) == 0:
    print("OK: grain holds — one row per (date, client, content)")
else:
    print(dup_keys[:5])


In [ ]:
# Query 2: Counts + date span
from datasets import load_dataset
import pandas as pd

fact_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

total = 0
dates = []
contents = set()
clients = set()

for row in fact_ds:
    if row.get('month') != '2026-03':
        continue
    total += 1
    dates.append(row['report_date'])
    contents.add(row['content_hash_id'])
    clients.add(row['client_hash_id'])

print("Counts + date span for month=2026-03:")
print(f"  total_rows: {total}")
print(f"  min_date: {min(dates) if dates else 'N/A'}")
print(f"  max_date: {max(dates) if dates else 'N/A'}")
print(f"  unique_contents: {len(contents)}")
print(f"  unique_clients: {len(clients)}")


In [ ]:
# Query 3: Availability filter with IS TRUE
from datasets import load_dataset

fact_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

total = 0
ga4_ok = 0
gsc_ok = 0
both_ok = 0

for row in fact_ds:
    if row.get('month') != '2026-03':
        continue
    total += 1
    ga4 = row.get('ga4_data_available')
    gsc = row.get('gsc_data_available')
    if ga4 is True:
        ga4_ok += 1
    if gsc is True:
        gsc_ok += 1
    if ga4 is True and gsc is True:
        both_ok += 1

print("Availability filter (IS TRUE) for month=2026-03:")
print(f"  total_rows: {total}")
print(f"  ga4_available: {ga4_ok}")
print(f"  gsc_available: {gsc_ok}")
print(f"  both_available: {both_ok}")
print(f"\\nSurvival rate with both flags IS TRUE: {both_ok}/{total} = {both_ok/total:.1%}")


## 4. Five features + "knowable at decision moment"

Feature frame built from the 90-day feature window (Dec 2025–Feb 2026) for content items present in March 2026.

| Feature | Knowable at decision moment because… |
|---------|--------------------------------------|
| `log_impressions_90d` | Sum of daily impressions over 90 days **before** decision date (Dec–Feb) |
| `avg_position_90d` | Mean `gsc_avg_position` (excluding 0 = no data) over prior 90 days |
| `ctr_90d` | `clicks_90d / impressions_90d * 100` from feature window only |
| `days_since_last_update` | From `dim_content` — static metadata, known at content creation |
| `content_age_days` | `(decision_date - content_created_at).days` — fully known at decision time |

In [ ]:
# Build 5-feature frame from 90-day feature window (Dec 2025 – Feb 2026)
# We aggregate daily fact for content items that appear in March 2026
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load March 2026 content IDs
fact_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

march_contents = set()
for row in fact_ds:
    if row.get('month') == '2026-03':
        march_contents.add((row['content_hash_id'], row['client_hash_id']))

print(f"Unique content items in March 2026: {len(march_contents)}")

# Load dim_content for metadata
dim_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

dim_map = {}
for row in dim_ds:
    if (row['content_hash_id'], row['client_hash_id']) in march_contents:
        dim_map[(row['content_hash_id'], row['client_hash_id'])] = row
    if len(dim_map) >= len(march_contents):
        break

# Aggregate feature window (Dec 2025 - Feb 2026) for March content
fact_ds2 = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

from collections import defaultdict
agg = defaultdict(lambda: {
    'impressions': 0, 'clicks': 0, 'sessions': 0,
    'positions': [], 'engagement_rates': []
})

for row in fact_ds2:
    key = (row['content_hash_id'], row['client_hash_id'])
    if key not in march_contents:
        continue
    if row.get('month') == '2026-03':
        continue  # skip target month
    if row.get('report_date') < '2025-12-01' or row.get('report_date') >= '2026-03-01':
        continue
    if not row.get('gsc_data_available', False):
        continue
    
    a = agg[key]
    a['impressions'] += row.get('gsc_impressions', 0) or 0
    a['clicks'] += row.get('gsc_clicks', 0) or 0
    a['sessions'] += row.get('ga4_sessions', 0) or 0
    pos = row.get('gsc_avg_position')
    if pos and pos > 0:
        a['positions'].append(pos)
    eng = row.get('ga4_engagement_rate')
    if eng is not None:
        a['engagement_rates'].append(eng)

# Build feature rows
rows = []
for (cid, clid), a in agg.items():
    if a['impressions'] == 0:
        continue
    dim = dim_map.get((cid, clid), {})
    rows.append({
        'content_hash_id': cid,
        'client_hash_id': clid,
        'impressions_90d': a['impressions'],
        'clicks_90d': a['clicks'],
        'sessions_90d': a['sessions'],
        'avg_position_90d': np.mean(a['positions']) if a['positions'] else None,
        'engagement_rate_90d': np.mean(a['engagement_rates']) if a['engagement_rates'] else None,
        'days_since_last_update': dim.get('days_since_last_update'),
        'word_count': dim.get('word_count'),
        'content_created_at': dim.get('content_created_at'),
    })

feat_df = pd.DataFrame(rows)
print(f"Aggregated feature rows: {len(feat_df)}")

# Compute derived features
feat_df["log_impressions_90d"] = np.log1p(feat_df["impressions_90d"])
feat_df["ctr_90d"] = np.where(feat_df["impressions_90d"] > 0,
                                 feat_df["clicks_90d"] / feat_df["impressions_90d"] * 100, 0)
feat_df["has_word_count"] = feat_df["word_count"].notna().astype(int)
feat_df["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_created_at"])).dt.days

# Label placeholder (trend_direction not in warehouse)
feat_df["is_declining_label"] = 0

print("Feature frame (first 10 rows):")
show_cols = ["content_hash_id", "client_hash_id",
             "log_impressions_90d", "avg_position_90d", "ctr_90d",
             "days_since_last_update", "content_age_days",
             "word_count", "has_word_count",
             "engagement_rate_90d", "sessions_90d",
             "is_declining_label"]
print(feat_df[show_cols].head(10).to_string(index=False))
print(f"\\nShape: {feat_df.shape}")


## 5. The trap — deliberate leakage experiment

1. Add `trend_pct` (label-derived) as a "feature"
2. Fit DecisionTreeClassifier (depth=2) → precision@50 jumps toward 1.0
3. **Remove** leak column → retrain → precision drops to honest level
4. Keep honest number; note the lesson

In [ ]:
# Deliberate leakage experiment (uses feature frame from Section 4)
# Reload the feature frame (or keep feat_df from Section 4)
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Recreate feat_df quickly (same as Section 4)
fact_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

march_contents = set()
for row in fact_ds:
    if row.get('month') == '2026-03':
        march_contents.add((row['content_hash_id'], row['client_hash_id']))

dim_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

dim_map = {}
for row in dim_ds:
    if (row['content_hash_id'], row['client_hash_id']) in march_contents:
        dim_map[(row['content_hash_id'], row['client_hash_id'])] = row
    if len(dim_map) >= len(march_contents):
        break

fact_ds2 = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

from collections import defaultdict
agg = defaultdict(lambda: {
    'impressions': 0, 'clicks': 0, 'sessions': 0,
    'positions': [], 'engagement_rates': []
})

for row in fact_ds2:
    key = (row['content_hash_id'], row['client_hash_id'])
    if key not in march_contents:
        continue
    if row.get('month') == '2026-03':
        continue
    if row.get('report_date') < '2025-12-01' or row.get('report_date') >= '2026-03-01':
        continue
    if not row.get('gsc_data_available', False):
        continue
    a = agg[key]
    a['impressions'] += row.get('gsc_impressions', 0) or 0
    a['clicks'] += row.get('gsc_clicks', 0) or 0
    a['sessions'] += row.get('ga4_sessions', 0) or 0
    pos = row.get('gsc_avg_position')
    if pos and pos > 0:
        a['positions'].append(pos)
    eng = row.get('ga4_engagement_rate')
    if eng is not None:
        a['engagement_rates'].append(eng)

rows = []
for (cid, clid), a in agg.items():
    if a['impressions'] == 0:
        continue
    dim = dim_map.get((cid, clid), {})
    rows.append({
        'content_hash_id': cid,
        'client_hash_id': clid,
        'impressions_90d': a['impressions'],
        'clicks_90d': a['clicks'],
        'sessions_90d': a['sessions'],
        'avg_position_90d': np.mean(a['positions']) if a['positions'] else None,
        'engagement_rate_90d': np.mean(a['engagement_rates']) if a['engagement_rates'] else None,
        'days_since_last_update': dim.get('days_since_last_update'),
        'word_count': dim.get('word_count'),
        'content_created_at': dim.get('content_created_at'),
    })

feat_df = pd.DataFrame(rows)
feat_df["log_impressions_90d"] = np.log1p(feat_df["impressions_90d"])
feat_df["ctr_90d"] = np.where(feat_df["impressions_90d"] > 0,
                                 feat_df["clicks_90d"] / feat_df["impressions_90d"] * 100, 0)
feat_df["has_word_count"] = feat_df["word_count"].notna().astype(int)
feat_df["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_created_at"])).dt.days

# Synthetic label: low impressions in feature window (since trend_direction not in warehouse)
y = (feat_df["impressions_90d"] < feat_df["impressions_90d"].median()).astype(int)

clean_cols = ["log_impressions_90d", "avg_position_90d", "ctr_90d",
             "days_since_last_update", "content_age_days",
             "word_count", "has_word_count",
             "engagement_rate_90d", "sessions_90d"]

X_clean = feat_df[clean_cols].fillna(0)

# Client-grouped split
clients = feat_df["client_hash_id"].unique()
np.random.seed(42)
test_clients = set(np.random.choice(clients, size=max(1, len(clients)//5), replace=False))
test_mask = feat_df["client_hash_id"].isin(test_clients)

X_train, X_test = X_clean[~test_mask], X_clean[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

# --- Clean model ---
clf_clean = DecisionTreeClassifier(max_depth=2, random_state=42, min_samples_leaf=100)
clf_clean.fit(X_train, y_train)
proba_clean = clf_clean.predict_proba(X_test.fillna(0))[:, 1]
top50_clean = pd.Series(proba_clean, index=y_test.index).nlargest(min(50, len(y_test))).index
prec50_clean = y_test.loc[top50_clean].mean()

# --- LEAKY model: add label as "trend_pct" feature ---
X_leaky = X_clean.copy()
X_leaky["trend_pct_LEAK"] = y  # directly leak the label

X_train_leaky, X_test_leaky = X_leaky[~test_mask], X_leaky[test_mask]
clf_leaky = DecisionTreeClassifier(max_depth=2, random_state=42, min_samples_leaf=100)
clf_leaky.fit(X_train_leaky, y_train)
proba_leaky = clf_leaky.predict_proba(X_test_leaky.fillna(0))[:, 1]
top50_leaky = pd.Series(proba_leaky, index=y_test.index).nlargest(min(50, len(y_test))).index
prec50_leaky = y_test.loc[top50_leaky].mean()

print(f"Clean model precision@50: {prec50_clean:.3f}")
print(f"Leaky model precision@50:  {prec50_leaky:.3f}")
print(f"\\nLeakage inflated precision by: {prec50_leaky - prec50_clean:.3f}")
print("\\nLesson: Adding label-derived columns (trend_pct, trend_direction, target-window metrics)")
print("makes scores look perfect but the model learns the label, not the signal.")
print("Removed leak column — keeping honest precision.")


## 6. One named limitation

**Limitation: Unbalanced panel / per-client history depth.**

The warehouse spans 2025-01-27 → 2026-06-30, but `dim_clients.gsc_data_start` varies widely — some clients have 17 months of history, others only 3. A fixed 90-day calendar window (Dec 2025–Feb 2026) includes zero-filled GA4 rows for clients whose `ga4_data_start` is later (flagged `ga4_data_available = FALSE`). Those zeros mean "no tracking yet", not "no engagement". This limits the usable client set for a fixed calendar window and biases features toward clients with longer history.

A stronger contract would use **per-client windows** anchored to each client's `gsc_data_start` / `ga4_data_start` rather than one global calendar window.

In [ ]:
# Evidence: dim_clients gsc_data_start distribution
from datasets import load_dataset
import pandas as pd

dim_clients_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

client_rows = []
for row in dim_clients_ds:
    client_rows.append({
        'gsc_data_start': row.get('gsc_data_start'),
        'ga4_data_start': row.get('ga4_data_start'),
    })

client_dist = pd.DataFrame(client_rows)
client_dist = client_dist.groupby(['gsc_data_start', 'ga4_data_start']).size().reset_index(name='client_count')
client_dist = client_dist.sort_values('gsc_data_start')

print("Client history start dates (dim_clients):")
print(client_dist.to_string(index=False))

print(f"\\nClients with GSC start before 2025-12-01 (enough for 90d feature window):")
early = client_dist[pd.to_datetime(client_dist['gsc_data_start']) < '2025-12-01']
print(f"  {early['client_count'].sum()} / {client_dist['client_count'].sum()} clients")

print(f"Clients with GA4 start before 2025-12-01:")
early_ga4 = client_dist[pd.to_datetime(client_dist['ga4_data_start']) < '2025-12-01']
print(f"  {early_ga4['client_count'].sum()} / {client_dist['client_count'].sum()} clients")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.